# EMOS Coding Lab: Automatic Coding with Text Similarity

In this lab we build a small automatic coding system step by step.

The task is simple to state: given a short text query, suggest the most appropriate classification code.

We will focus on transparency rather than performance. At each step we ask: what information do we have, what transformation are we applying, and how can we validate the result?

## 1. Introduction

**Automatic coding** means assigning a classification code to a text description. In Official Statistics, this can support tasks such as coding economic activities, occupations, products, or causes of death.

A useful automatic coding system should not only output a code. It should also help us understand why that code was suggested, how confident the system is, and where human validation is needed.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

import re
from collections import Counter

import pandas as pd
import matplotlib.pyplot as plt

from src.data_loading import load_labelled_queries, load_ateco_classification, load_ateco_descriptors
from src.descriptors import build_descriptors
from src.embeddings import compute_tfidf_embeddings
from src.embeddings import compute_centroid_embeddings, load_embedding_model, compute_sentence_embeddings
from src.similarity import compute_similarity_matrix, get_top_k_predictions, add_predictions_to_queries
from src.evaluation import summarize_accuracy, get_error_examples

## 2. Data loading

We use labelled queries as examples of texts that need to be coded. Each query already has a true code, so we can evaluate our automatic suggestions.

For the classroom exercise we use a shareable labelled dataset prepared for the lab. The confidential full dataset is not needed: the goal is to understand the coding workflow step by step.


In [ ]:
raw_queries = load_labelled_queries()
raw_queries.head()

In [ ]:
raw_queries.shape, raw_queries["true_code"].nunique()

## 3. First look at the data

Before building an automatic coding model, we should understand the data. Simple checks often reveal important issues: which areas of the classification are common, which words dominate the text, and how much noise appears in the vocabulary.


### Distribution of 2-digit codes

The first two digits identify a broad division of the classification. Looking at this distribution helps us understand whether the labelled data is balanced or concentrated in a few areas.


In [ ]:
code_distribution = (
    raw_queries.assign(division=raw_queries["true_code"].str[:2])
    .groupby("division")
    .size()
    .reset_index(name="n_queries")
    .sort_values("n_queries", ascending=False)
    .reset_index(drop=True)
)

code_distribution.head(15)


In [ ]:
top_divisions = code_distribution.head(15)

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(top_divisions["division"], top_divisions["n_queries"], color="#4C956C")
ax.set_xlabel("2-digit code")
ax.set_ylabel("Number of queries")
ax.set_title("Most frequent 2-digit codes")
plt.show()


### Token frequencies

A token is a word-like unit extracted from text. Here we use a slightly more careful tokenizer than a simple `split()`:

- lowercase the text
- keep letters, accented Italian letters, and digits
- remove common Italian stopwords
- remove one-character tokens

This is still a simple approach, but it is closer to what we would do with real administrative text.


In [ ]:
ITALIAN_STOPWORDS = {
    "a", "ad", "al", "allo", "ai", "agli", "all", "alla", "alle", "anche",
    "che", "chi", "ci", "coi", "col", "come", "con", "contro", "cui",
    "da", "dal", "dallo", "dai", "dagli", "dall", "dalla", "dalle",
    "de", "dei", "del", "dello", "degli", "dell", "della", "delle",
    "di", "dove", "e", "ed", "era", "erano", "essere", "fa", "fra",
    "gli", "ha", "hai", "hanno", "ho", "i", "il", "in", "io", "la",
    "le", "lei", "li", "lo", "loro", "lui", "ma", "mi", "nel",
    "nello", "nei", "negli", "nell", "nella", "nelle", "no", "non",
    "o", "per", "piu", "più", "quale", "quali", "quando", "quanto",
    "quella", "quelle", "quelli", "quello", "quest", "questa", "queste",
    "questi", "questo", "se", "sei", "si", "sia", "sono", "su",
    "sul", "sullo", "sui", "sugli", "sull", "sulla", "sulle", "tra",
    "tu", "un", "una", "uno", "vi", "è",
}

token_pattern = re.compile(r"[a-zàèéìòù0-9]+", flags=re.IGNORECASE)


def tokenize(text):
    """Returns informative lowercase tokens from a text string."""

    tokens = token_pattern.findall(str(text).lower())
    return [
        token
        for token in tokens
        if token not in ITALIAN_STOPWORDS and len(token) > 1
    ]


token_df = raw_queries[["query", "true_code"]].copy().reset_index(drop=True)
token_df["query_id"] = token_df.index
token_df["code_2"] = token_df["true_code"].str[:2]
token_df["code_4"] = token_df["true_code"].str[:5]
token_df["code_5"] = token_df["true_code"]
token_df["tokens"] = token_df["query"].apply(tokenize)

token_occurrences = (
    token_df[["query_id", "code_2", "tokens"]]
    .explode("tokens")
    .dropna(subset=["tokens"])
    .rename(columns={"tokens": "token"})
)

total_tokens = len(token_occurrences)

token_frequency = (
    token_occurrences["token"]
    .value_counts()
    .rename_axis("token")
    .reset_index(name="token_frequency")
)

document_frequency = (
    token_occurrences.drop_duplicates(["query_id", "token"])["token"]
    .value_counts()
    .rename_axis("token")
    .reset_index(name="document_frequency")
)

vocabulary = token_frequency.merge(document_frequency, on="token", how="left")

vocabulary_summary = pd.DataFrame(
    [
        {"metric": "total_tokens", "value": total_tokens},
        {"metric": "unique_tokens", "value": len(vocabulary)},
        {
            "metric": "hapax_tokens",
            "value": int(vocabulary["token_frequency"].eq(1).sum()),
        },
        {
            "metric": "hapax_share_of_vocabulary",
            "value": float(vocabulary["token_frequency"].eq(1).mean()),
        },
    ]
)

vocabulary_summary


In [ ]:
top_tokens_by_frequency = vocabulary.sort_values(
    "token_frequency",
    ascending=False,
).head(20)

top_tokens_by_frequency


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(
    top_tokens_by_frequency["token"][::-1],
    top_tokens_by_frequency["token_frequency"][::-1],
    color="#3A7CA5",
)
ax.set_xlabel("Token frequency")
ax.set_title("Top 20 tokens in labelled queries")
plt.show()


### Document frequency

Token frequency counts how many times a token appears in total. **Document frequency** counts in how many different queries the token appears.

This distinction matters for TF-IDF: a token that appears in many queries is usually less informative than a token concentrated in fewer queries.


In [ ]:
top_tokens_by_document_frequency = vocabulary.sort_values(
    "document_frequency",
    ascending=False,
).head(20)

top_tokens_by_document_frequency


### Hapax and spelling noise

An **hapax** is a token that appears only once. Hapax are not always errors: they can be rare but meaningful words. In real text data, however, many hapax are spelling mistakes, abbreviations, or very specific names.

For TF-IDF this matters because every distinct token adds one dimension to the vocabulary. Many noisy hapax can make the vectors larger and less informative.


In [ ]:
hapax_tokens = set(vocabulary.loc[vocabulary["token_frequency"].eq(1), "token"])

hapax_token_examples = (
    token_occurrences.loc[
        token_occurrences["token"].isin(hapax_tokens),
        ["token", "query_id", "code_2"],
    ]
    .merge(
        token_df[["query_id", "query", "code_4", "code_5"]],
        on="query_id",
        how="left",
    )
    .assign(
        token_length=lambda data: data["token"].str.len(),
        contains_digit=lambda data: data["token"].str.contains(r"\d", regex=True),
    )
    .sort_values(["token", "code_5"])
    .reset_index(drop=True)
)

hapax_token_examples.head(30)


## 4. Classification and descriptors

A classification system is a structured list of possible codes. A code is not just a number: it represents a statistical category.

Before using the classification for automatic coding, we inspect its structure. ATECO is hierarchical: broad sections and divisions appear at higher levels, while more detailed activity codes appear lower in the hierarchy.


In [ ]:
ateco_classification = load_ateco_classification()
ateco_classification.head()


In [ ]:
hierarchy_labels = {
    1: "section",
    2: "division (2-digit)",
    3: "group (3-digit)",
    4: "class (4-digit)",
    5: "target code (5-digit)",
    6: "detailed code (6-digit)",
}

classification_summary = (
    ateco_classification.assign(
        hierarchy_label=ateco_classification["GERARCHIA"].map(hierarchy_labels)
    )
    .groupby(["GERARCHIA", "hierarchy_label"])["CODICE"]
    .nunique()
    .reset_index(name="n_codes")
    .sort_values("GERARCHIA")
    .reset_index(drop=True)
)

classification_summary


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(
    classification_summary["hierarchy_label"],
    classification_summary["n_codes"],
    color="#6C5B7B",
)
ax.set_ylabel("Number of distinct codes")
ax.set_title("ATECO hierarchy levels")
ax.tick_params(axis="x", rotation=30)
plt.show()


The official classification file contains the hierarchy and official notes. For text similarity, we use a processed descriptor file where detailed notes have been split into shorter snippets.

This creates several descriptor snippets for the same 5-digit target code, which is why the notebook offers the `CONCAT` and `CENTROID` strategies below.


In [ ]:
DESCRIPTOR_METHOD = "CONCAT"  # Try also: "CENTROID"

raw_descriptors = load_ateco_descriptors()
descriptor_table = build_descriptors(raw_descriptors, method=DESCRIPTOR_METHOD)

descriptor_table.head()

In [ ]:
descriptor_table.shape

## 5. Embeddings

An **embedding** is a numerical representation of text. Intuitively, it places texts in a space where texts with similar meaning should be closer together.

Many modern embedding models are shared through **Hugging Face**, an online hub for machine learning models, datasets, and demos. In practice, this means we do not need to train a language model from scratch: we can reuse a pre-trained multilingual model and apply it to our coding task.

For reliability in a classroom setting, we start with a simple TF-IDF representation. It is not a modern semantic embedding model, but it makes the same similarity workflow visible and runs quickly.

If the sentence-transformers model is installed and available, the optional cell below switches to a multilingual semantic embedding model from Hugging Face.

### TF-IDF in plain language

Before using large embedding models, we start with **TF-IDF**: Term Frequency-Inverse Document Frequency.

The idea is to transform text into numbers using a shared vocabulary. We first build **one common vocabulary** from all texts we want to compare: the labelled queries and the target code descriptors. If the vocabulary contains 10,000 distinct words, then every query and every descriptor is represented as a vector with 10,000 positions.

A very simple text vector could contain only `0` and `1`:

- `1` means the word is present in that text
- `0` means the word is absent

TF-IDF is a slightly richer version of this. Instead of only saying whether a word is present, it gives more weight to words that are useful for distinguishing one text from another.

For example, a common word that appears almost everywhere receives little weight. A more specific word that appears in fewer texts receives more weight.

So the important point is: **query vectors and descriptor vectors have the same dimension**, because they are built using the same vocabulary. This makes cosine similarity possible.


In [ ]:
query_vectors, descriptor_vectors = compute_tfidf_embeddings(
    raw_queries["query"],
    descriptor_table["descriptor_text"],
    tokenizer=tokenize,
)

if DESCRIPTOR_METHOD == "CENTROID":
    target_codes, target_vectors = compute_centroid_embeddings(
        descriptor_vectors,
        descriptor_table["code"],
    )
else:
    target_codes = descriptor_table["code"].tolist()
    target_vectors = descriptor_vectors

query_vectors.shape, target_vectors.shape, len(target_codes)

In [ ]:
# Optional: use a multilingual sentence embedding model instead of TF-IDF.
# The selected model is downloaded from Hugging Face the first time it is used.
# Larger models may need more memory, more time, or a GPU/cloud environment.

# EMBEDDING_MODELS = [
#     {
#         "model_id": "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
#         "trust_remote_code": False,
#         "note": "small classroom default",
#     },
#     {
#         "model_id": "BAAI/bge-m3",
#         "trust_remote_code": False,
#         "note": "strong multilingual embedding model",
#     },
#     {
#         "model_id": "intfloat/multilingual-e5-base",
#         "trust_remote_code": False,
#         "note": "multilingual E5 base model",
#     },
#     {
#         "model_id": "Alibaba-NLP/gte-multilingual-base",
#         "trust_remote_code": True,
#         "note": "requires trusting custom model code",
#     },
#     {
#         "model_id": "Qwen/Qwen3-Embedding-4B",
#         "trust_remote_code": True,
#         "note": "larger model; better suited to GPU/cloud testing",
#     },
# ]

# selected_model = EMBEDDING_MODELS[0]
# selected_model

In [ ]:
# Uncomment this cell to replace TF-IDF vectors with sentence embeddings.

# model = load_embedding_model(
#     selected_model["model_id"],
#     trust_remote_code=selected_model["trust_remote_code"],
# )
# query_vectors = compute_sentence_embeddings(raw_queries["query"], model)
# descriptor_vectors = compute_sentence_embeddings(descriptor_table["descriptor_text"], model)

# if DESCRIPTOR_METHOD == "CENTROID":
#     target_codes, target_vectors = compute_centroid_embeddings(
#         descriptor_vectors,
#         descriptor_table["code"],
#     )
# else:
#     target_codes = descriptor_table["code"].tolist()
#     target_vectors = descriptor_vectors

## 6. Similarity-based coding

Cosine similarity compares the direction of two vectors. In this lab, a high value means that a query and a target code descriptor use similar language or have similar meaning.

For each query, we rank all candidate codes by similarity and keep the best suggestions.

In [ ]:
similarity_matrix = compute_similarity_matrix(query_vectors, target_vectors)

top_k = get_top_k_predictions(
    similarity_matrix,
    target_codes,
    k=5,
)

predictions = add_predictions_to_queries(raw_queries, top_k)
predictions[["query", "true_code", "predicted_code", "confidence", "top_codes"]].head(10)

## 7. Evaluation

**Top-1 accuracy** asks: did the first suggested code match the labelled code?

**Top-k accuracy** asks: was the labelled code somewhere in the first k suggestions?

Top-k accuracy is important in Official Statistics because an automatic system may support a human coder by producing a shortlist rather than making a final decision alone.

In [ ]:
accuracy_table = summarize_accuracy(predictions, ks=(1, 3, 5))
accuracy_table

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))

ax.bar(
    accuracy_table["metric"],
    accuracy_table["value"],
    color="#3A7CA5",
)

ax.set_ylim(0, 1)
ax.set_ylabel("Accuracy")
ax.set_title("Top-k accuracy")

# Showing percentages makes the metric easier to read at a glance.
for index, value in enumerate(accuracy_table["value"]):
    ax.text(index, value + 0.02, f"{value:.1%}", ha="center")

plt.show()

## 8. Error analysis

Accuracy gives a summary, but error analysis teaches us more. We inspect cases where the labelled code was not the first suggestion.

For each error, ask: is the query ambiguous, is the descriptor too general, or are the two target codes genuinely similar?

In [ ]:
get_error_examples(predictions, k=1, n=15)

In [ ]:
# Cases where the first suggestion is wrong, but the correct code appears in the Top-5.
almost_right = predictions[
    (predictions["true_code"] != predictions["predicted_code"])
    & (predictions.apply(lambda row: row["true_code"] in row["top_codes"][:5], axis=1))
]

almost_right[["query", "true_code", "predicted_code", "top_codes"]].head(15)

## 9. Discussion: Official Statistics perspective

This lab shows the core mechanism behind similarity-based automatic coding. The workflow is transparent:

1. Define the target classification.
2. Describe each target code with text.
3. Represent queries and target codes as vectors.
4. Compare them with similarity.
5. Suggest the closest codes.
6. Evaluate and inspect errors.

In production, we would need more checks: data quality controls, monitoring over time, careful validation by domain experts, and clear rules for low-confidence cases.

The important idea is not that AI replaces statistical expertise. It can help organize work, produce candidate codes, and highlight uncertain cases where human judgment is most valuable.